# Bioinvasion analysis: demo

Runs the analysis of Rohrbach et al. (2025) on the demo project `examples/isodontia` (*Isodontia mexicana* in Europe, GBIF data). The same functions are used by the command line (`cli.py`) and the web app. Settings are in `examples/isodontia/project.toml`; results are written into that folder.

Every stage is skipped if its inputs and settings have not changed, so re-running cells is cheap.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # repository root

from IPython.display import Image, Markdown, display
from bioinvasion import pipeline
from bioinvasion.config import load_config
from bioinvasion.report import insights

cfg = load_config('../examples/isodontia')

## 1. Observations and least-cost paths
Imports the observations, removes those the analysis cannot use, keeps the earliest observation per cost surface cell and connects each observation to the nearest earlier one.

In [ ]:
paths_dir = pipeline.compute_paths(cfg)
run = pipeline.load_paths(paths_dir)
run['manifest']['results']

## 2. Sensitivity sweep
An overview over the whole range of accumulated costs, then a high-resolution sweep over the suggested range. The rule behind the suggestion is printed.

In [ ]:
coarse, fine, chosen = pipeline.sweep(cfg)
display(Markdown(chosen['rule']))
d = insights.nearest_definition(fine, cfg.delineation.robust_definition)
insights.stable_ranges(fine, d)

## 3. Populations and expansion rates
Choose the threshold in `project.toml` (`[delineation] threshold`) or pass it here.

In [ ]:
out_dir, summary, checks = pipeline.report(cfg, threshold=cfg.delineation.threshold)
print(summary)
display(Markdown('\n'.join(f'- {text}' for level, text in checks)))

In [ ]:
for name in sorted(os.listdir(out_dir)):
    if name.endswith('.png'):
        display(Image(os.path.join(out_dir, name), width=800))